# Exploratory Data Analysis

Packages

In [ ]:
# Public packages
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from tqdm import tqdm
import math
import os
import re
from pathlib import Path

# Custom packages
from tools.filter import FilterDF as fdf
from tools.benchmarks import ParetoAnalysis as pa
from tools.benchmarks import AccuracyCalculation as ac
from tools.integrity_fixes import DataFixer as fix, DataExporter as exporter

# Set directory to project root
def find_project_root(start: Path = Path().absolute()) -> Path:
    for parent in start.parents:
        if (parent / "requirements.txt").exists(): return parent
    return start 
os.chdir(find_project_root())

# Preemptively set new Pandas option, also set matplotlib to close
pd.options.mode.copy_on_write = True
%matplotlib inline
%config InlineBackend.close_figures=True

# Allow reloading of custom Python classes without resetting kernel
pd.set_option('display.max_rows', 100)
%load_ext autoreload
%autoreload 2

Read data from parquet files

In [ ]:
# Load formatted data
%store -r static_data_merged
%store -r sales_data_merged

# Check if the data is already imported
if 'static_data_merged' not in locals():

    # Retrieve filnames
    static_data_filenames = os.listdir(f"2_palate_data_parquet_cleaned")

    # Initialize a dictionary for the four parquet files of "static reference data"
    static_data = {}
    for filename in tqdm(static_data_filenames):

        # Exclude other files and folders
        if ".parquet" in filename:
            df = pd.read_parquet(f"2_palate_data_parquet_cleaned/{filename}")
            base_name = re.sub(r'\.parquet$', '', filename)
            static_data[base_name] = df

    # Rename and store
    static_data_merged = static_data.copy()
    
# Data already exists
else:
    static_data = static_data_merged.copy()

%store static_data_merged

# Check if the data is already imported
if 'sales_data_merged' not in locals():

    # Retrieve filnames
    restaurant_sales_filenames = os.listdir(f"2_palate_data_parquet_cleaned/orders_item_level")

    # Initialize a dictionary for the 30 parquet files of "restaurant sales data"
    sales_and_menu_data = {}
    for filename in tqdm(restaurant_sales_filenames):
        
        # Exclude other files and folders
        if ".parquet" in filename:
            df = pd.read_parquet(f"2_palate_data_parquet_cleaned/orders_item_level/{filename}")
            location_id = re.sub(r'_sales_and_menu\.parquet$', '', filename)
            sales_and_menu_data[location_id] = df
    
    # Rename and store
    sales_data_merged = {}
    for loc_id, df in sales_and_menu_data.items():
        sales_data_merged[loc_id] = df.copy()
    
# Data already exists
else:
    sales_and_menu_data = {}
    for loc_id, df in sales_data_merged.items():
        sales_and_menu_data[loc_id] = df.copy()

%store sales_data_merged

# Rename data for ease of use
before_after_details = static_data['before_after_details'].copy() # Promotional items (30 rows)
customers = static_data['customers'].copy() # Specific customer information: for matching customers with orders
items_tagged = static_data['items_tagged'].copy() # Menu items for all restaurants: for matching plant-based labels with orders
locations = static_data['locations'].copy() # Restaurant details (30 rows)
location_ids = list(sales_and_menu_data.keys())

# True promos
before_after_details_true = pd.read_csv('data/before_after_details_true.csv', index_col='location_id')

# Timezones
timezones = pd.read_csv('data/timezones.csv', index_col='location_id')['timezone'].to_dict()
for loc_id, df in sales_and_menu_data.items():
    df.index = df.index.tz_convert(timezones[loc_id])
    sales_and_menu_data[loc_id] = df

# import from pickle
time_differences = pd.read_pickle('data/2_palate_data_parquet_cleaned/time_differences.pkl')
time_differences_details = pd.read_pickle('data/2_palate_data_parquet_cleaned/time_differences_details.pkl')

restaurants_by_4m_coverage = pd.read_csv('data/2_palate_data_parquet_cleaned/restaurants_by_4m_coverage.csv')['location_id'].tolist()

## Static Reference Data Exploration

### Menu Stats

In [ ]:
print(f'Total number of menu items: {items_tagged.shape[0]}')
print(f'Median number of menu items for a restaurant: {round(items_tagged.groupby("location_id")["item_name"].count().median())}')

### Menu Visuals

In [ ]:
# How many categories to include?
topn = 20

# Irrelevant dish categories
irrelevant = 2

## Colors for the bars
colors1 = ["#ef8a62", "#67a9cf"]
colors2 = ["#ff6961", "#aec6cf", "#77dd77"]

# Calculate the counts for each combination
pivot_df1 = items_tagged.groupby(['is_plant_based', 'is_plant_based'], observed=True).size().unstack(fill_value=0) # is_alcohol should be the second group by
pivot_df1 = pivot_df1.loc[pivot_df1.sum(axis=1).sort_values(ascending=False).index] # Sort

# Calculate the counts for each combination
pivot_df2 = items_tagged.groupby(['item_type', 'is_plant_based'], observed=True).size().unstack(fill_value=0)
pivot_df2 = pivot_df2.loc[pivot_df2.sum(axis=1).sort_values(ascending=False).index] # Sort

# Calculate the counts for each combination
pivot_df3 = items_tagged.groupby(['dish_category', 'is_plant_based'], observed=True).size().unstack(fill_value=0)
pivot_df3 = pivot_df3.loc[pivot_df3.sum(axis=1).sort_values(ascending=False).index[:topn]] # Sort

# Calculate the counts for each combination
pivot_df4 = items_tagged.groupby(['dish_category', 'is_plant_based'], observed=True).size().unstack(fill_value=0)
pivot_df4 = pivot_df4.loc[pivot_df4.sum(axis=1).sort_values(ascending=False).index[irrelevant:topn+irrelevant]] # Sort

# Add to lists
dfs = [pivot_df1, pivot_df2, pivot_df3, pivot_df4]
color_groups = [colors1, colors2, colors2, colors2]
titles = ["Plant Based", "Meal Type Counts", "Dish Category Counts", "Dish Category Counts w/o Alcohol or Unknowns"]
xlabs = ["Plant Based", "Meal Type", "Dish Category", "Dish Category"]
legends = ["Is Alcohol?", "Is Plant Based?", "Is Plant Based?", "Is Plant Based?"]
coordinates = [(0,0),(0,1),(1,0),(1,1)]

# Create a 2x2 grid of subplots
fig, axes = plt.subplots(2, 2, figsize=(12, 10))

# Zip together for looping
data_for_visual = zip(dfs, color_groups, titles, xlabs, legends, coordinates)
for i, data_tuple in enumerate(data_for_visual):

    # Unpack
    df, colors, title, xlab, legend, coordinate = data_tuple
    a, b = coordinate

    # Initialize a series of zeros with the same index
    left_start = pd.Series(0, index=df.index)

    # Loop through every variable to be stacked
    for j, col in enumerate(df.columns):

        # Stack up multiple bar charts
        sns.barplot(x=df[col], 
                    y=df.index.tolist(), 
                    left=left_start, 
                    color=colors[j], 
                    label=col,
                    ax=axes[a,b])
        
        # Start the next level exactly where the last one
        left_start = left_start + df[col]

    axes[a,b].set_ylabel('Count')
    axes[a,b].set_xlabel(xlab)
    #axes[a,b].set_yticks(range(df.index.size))
    #axes[a,b].set_yticklabels(labels=df.index.to_series().str.capitalize().tolist())
    axes[a,b].set_title(title)
    axes[a,b].legend(title=legend)

# Adjust layout
plt.tight_layout()

# Create a margin for a global title
plt.subplots_adjust(top=0.88)

# Add global title
fig.suptitle('                   Visual Summary of Menu Items', fontsize=16)

# Save plot
plt.savefig('visuals/Menu Summary Stats.png', bbox_inches='tight')

# Show the plots
plt.show()

### Customer Stats

Merge in customer data

In [ ]:
# Initialize dict all data
sales_menu_customers_data = {}
for loc_id, df in sales_and_menu_data.items():

    # Prevent overwriting
    df = df.copy()

    # Keep the index, since merges don't keep it
    df.reset_index(inplace=True)

    # Combine
    merged = pd.merge(df, customers, 
                      on=['location_id', 'customer_id'], 
                      how='left')

    # Reset the index back to datetimes
    merged.set_index('created_at', inplace=True, drop=False) # Keep the column for later as well

    # Save
    sales_menu_customers_data[loc_id] = merged

Totals

In [ ]:
# Number of customers, number of customers with gender data, number of customers with age data
print(f'Total number of customers: {customers.shape[0]:,}')
print(f'Customers with gender data: {customers["gender"].notna().sum():,}')
print(f'Customers with age data: {customers["age"].notna().sum():,}')

In [ ]:
customer_id_count = 0
total_count = 0
max_pct = 0
min_pct = 1
for loc_id, df in sales_menu_customers_data.items():
    customer_id_count += df['customer_id'].notna().sum()
    total_count += df.shape[0]
    min_pct = min(min_pct, df['customer_id'].notna().sum()/df.shape[0])
    max_pct = max(max_pct, df['customer_id'].notna().sum()/df.shape[0])
    
    
print(f'Total number of customers in all restaurants: {customer_id_count:,}')
print(f'Total number of orders in all restaurants: {total_count:,}')
print(f'Percentage of customers with orders: {customer_id_count/total_count:.2%}')
print(min_pct)
print(max_pct)
# Check if the data is already imported
     

Intersecting customers bases

In [ ]:
# Precompute the set of customers in each restaurant for efficiency
precomputed_customers = {}
for loc_id in location_ids:
    precomputed_customers[loc_id] = set(fdf(customers).filter('location_id', loc_id)['customer_id'].unique().tolist())

# Restaurant 1
for i, loc_id1 in enumerate(location_ids):
    customer_set1 = precomputed_customers[loc_id1]

    # Restaurant 2
    for loc_id2 in location_ids[i:]:
        
        # As long as they're different
        if loc_id1 != loc_id2:
            customer_set2 = precomputed_customers[loc_id2]

            # Intersect
            intersection = customer_set1.intersection(customer_set2)

            # Result
            if intersection:
                print(loc_id1, loc_id2, len(intersection))
            else:
                pass # Don't print anything if the restaurants don't have an intersecting customer base

Female proportions

In [ ]:
# Initialize female proportions list
female_proportions_list = []
for loc_id, df in sales_menu_customers_data.items():
    print(loc_id)
    # Prevent overwriting
    df = df.copy()

    # Initialize summary row for resulting summary dataframe
    row = {'location_id': loc_id}

    # Promo date (normalize time zones)
    cross_over = before_after_details.loc[loc_id,'cross_over_date']
    #df.index = df.index.tz_localize(None)

    # Subset and aggregate the get the number of customers of genders
    before_genders = df.loc[:cross_over,'gender'].value_counts()
    after_genders = df.loc[cross_over:,'gender'].value_counts()
    
    
    # As long as they're nonempty, calculate the fractions
    if not before_genders.empty and not after_genders.empty:
        before_female_total = before_genders.loc['female']
        after_female_total = after_genders.loc['female']
        before_known_gender_total = before_genders.loc['male'] + before_genders.loc['female']
        after_known_gender_total = after_genders.loc['male'] + after_genders.loc['female']

        # No zero divisor allowed
        before_frac_female = -1
        if before_known_gender_total != 0:
            before_frac_female = before_female_total/before_known_gender_total
            
        # No zero divisor allowed
        after_frac_female = -1
        if after_known_gender_total != 0:
            after_frac_female = after_female_total/after_known_gender_total
        
        # Is it a large enough sample?
        large_enough = 1000
        sample_qualifer = ""
        if large_enough < before_known_gender_total and large_enough < after_known_gender_total:
            sample_qualifer = "Large Enough Sample"

        # Store in summary row
        row['b_f_frac'] = round(before_frac_female*100)/100
        row['a_f_frac'] = round(after_frac_female*100)/100
        row['enough_data'] = bool(sample_qualifer)
        row['b_notna_total'] = before_known_gender_total
        row['a_notna_total'] = after_known_gender_total


    # If both are empty, there's no data
    elif before_genders.empty and after_genders.empty:

        print(loc_id, "--No customer data!")

    # If one is empty, there's no comparison
    else:

        print(loc_id, "--Not enough data before or after.")

    # Save
    female_proportions_list.append(row)

female_proportions = pd.DataFrame(female_proportions_list)

View

In [ ]:
female_proportions

Revisiting customers

In [ ]:
def retrieve_ids(df, ids, j):
    if j == 10:
        ids.extend(df.index.tolist())
    return df

# Initialize summary list for revisiting customers
customer_revisit_row_list = []
dict_of_customer_ids = {}
for loc_id, df in sales_menu_customers_data.items():
    
    # # Prevent overwriting
    # df = df.copy()

    # # Need to group by aggregate the time series, 'created_at' but don't need to reset later because no changes will be made
    # df.reset_index(inplace=True)

    # Summary row
    row = {'location_id': loc_id, 'total': df['customer_id'].nunique()}

    # Did they revisit _x_ number of times?
    revisit_times = [1, 2, 5, 10]
    customer_ids = []
    for j in revisit_times:

        # For each customer, did they come at multiple times
        num_revisits = (df
                        .groupby('customer_id', observed=True)
                        ['created_at']
                        .nunique()
                        .to_frame('revisits')
                        .query('revisits > @j')
                        .pipe(retrieve_ids, customer_ids, j)
                        .shape[0])

        # Store in summary row
        row['more than ' + str(j)] = num_revisits

    # Save
    customer_revisit_row_list.append(row)

    dict_of_customer_ids[loc_id] = customer_ids
    
revisits = pd.DataFrame(customer_revisit_row_list)

In [ ]:
%store -r restaurants_by_4m_coverage
%store -r before_after_details_true

In [ ]:
fig, ax = plt.subplots(figsize=(14, 8))

for loc_id in restaurants_by_4m_coverage[:1]:
    
    df = sales_and_menu_data[loc_id]
    customers = dict_of_customer_ids[loc_id]
    promo_datetime = before_after_details_true.loc[loc_id,'cross_over_date'].tz_convert(None)
    
    customer_active_weeks_dict = {}
    for customer in customers[:60]:
        customer_active_weeks_dict[customer] = (df
                                                .query('customer_id == @ customer')
                                                .resample('W')
                                                .size()
                                                .to_frame(name='W')
                                                .query('0 < W')
                                                .index
                                                .tz_localize(None)
                                                .to_period('W')
                                                .tolist())
    
    for customer, active_weeks in customer_active_weeks_dict.items():
        
        # For every active week
        for week in active_weeks:

            # Place a blue dot
            ax.hlines(y=customer, xmin=week.start_time, xmax=week.end_time, colors='blue', lw=2, label=customer)

        # Place a red circle for the promotional item
        ax.plot(promo_datetime, customer, 'ro', alpha=0.5)
    
    ax.set_title(loc_id)

In [ ]:
fig, ax = plt.subplots(figsize=(14, 8))

for loc_id in restaurants_by_4m_coverage[1:2]:
    
    df = sales_and_menu_data[loc_id]
    customers = dict_of_customer_ids[loc_id]
    promo_datetime = before_after_details_true.loc[loc_id,'cross_over_date'].tz_convert(None)
    
    customer_active_weeks_dict = {}
    for customer in customers[:60]:
        customer_active_weeks_dict[customer] = (df
                                                .query('customer_id == @ customer')
                                                .resample('W')
                                                .size()
                                                .to_frame(name='W')
                                                .query('0 < W')
                                                .index
                                                .tz_localize(None)
                                                .to_period('W')
                                                .tolist())
    
    for customer, active_weeks in customer_active_weeks_dict.items():
        
        # For every active week
        for week in active_weeks:

            # Place a blue dot
            ax.hlines(y=customer, xmin=week.start_time, xmax=week.end_time, colors='blue', lw=2, label=customer)

        # Place a red circle for the promotional item
        ax.plot(promo_datetime, customer, 'ro', alpha=0.5)
    ax.set_title(loc_id)

In [ ]:
customer_repeat_list = []
for loc_id in restaurants_by_4m_coverage:

    df = sales_and_menu_data[loc_id]

    promo_datetime = before_after_details_true.loc[loc_id,'cross_over_date'].tz_convert(None)

    date_bounds = (df
                .reset_index()
                .groupby('customer_id', observed=True)
                ['created_at']
                .agg(['min', 'max'])
                .assign(min = lambda df: df['min'].dt.tz_localize(None),
                        max = lambda df: df['max'].dt.tz_localize(None))
    )

    customer_repeat_list.append({'loc_id':loc_id, 
                                 'total w/ before + after':(date_bounds['min'].lt(promo_datetime) & date_bounds['max'].gt(promo_datetime)).sum(), 
                                 'pct w/ before + after':np.round((date_bounds['min'].lt(promo_datetime) & date_bounds['max'].gt(promo_datetime)).mean(), 2)})

In [ ]:
pd.DataFrame(customer_repeat_list)

In [ ]:
from pandas import DateOffset

customer_4m_repeat_list = []
for loc_id in restaurants_by_4m_coverage:

    df = sales_and_menu_data[loc_id]
    promo_datetime = before_after_details_true.loc[loc_id, 'cross_over_date'].tz_convert(None)
    before_2_months = promo_datetime - DateOffset(months=2)
    after_2_months = promo_datetime + DateOffset(months=2)

    date_bounds = (df
                   .reset_index()
                   .groupby('customer_id', observed=True)
                   ['created_at']
                   .agg(list)
                   .apply(lambda dates: [d.tz_localize(None) for d in dates])
    )

    # Define conditions for at least one entry in the 2 months before and 2 months after
    has_before_entry = date_bounds.apply(lambda dates: any(before_2_months <= d < promo_datetime for d in dates))
    has_after_entry = date_bounds.apply(lambda dates: any(promo_datetime < d <= after_2_months for d in dates))

    # Count customers meeting both conditions
    condition = has_before_entry & has_after_entry
    total_customers = condition.sum()
    percent_customers = np.round(condition.mean(), 4)  # Convert to percentage

    customer_4m_repeat_list.append({'loc_id':loc_id, 
                                     'total w/ before + after':total_customers, 
                                     'pct w/ before + after':percent_customers })


In [ ]:
pd.DataFrame(customer_4m_repeat_list)

In [ ]:
for loc_id in restaurants_by_4m_coverage:

    df = sales_and_menu_data[loc_id]

    date_bounds = (df
                .reset_index()
                .groupby('customer_id', observed=True)
                ['created_at']
                .agg(['min', 'max'])
                .assign(min = lambda df: df['min'].dt.tz_localize(None),
                        max = lambda df: df['max'].dt.tz_localize(None))
    )

    print(loc_id, (date_bounds['min'].lt(promo_datetime) & date_bounds['max'].gt(promo_datetime)).mean())

In [ ]:
date_bounds

View

In [ ]:
revisits.set_index('location_id').loc[restaurants_by_4m_coverage]

## Restaurant Sales Data Exploration

Totals

In [ ]:
# Recheck total number of entries
sales_items_total = 0
sales_data_total = 0
sales_transactions_total = 0
for loc_id, df in sales_and_menu_data.items():
    sales_items_total += df['item_quantity'].sum()
    sales_data_total += df.shape[0]
    sales_transactions_total += df['order_id'].nunique()

# Print
print(f'Total number of items sold: {sales_items_total:,}')
print(f'Total number of sales entries (row): {sales_data_total:,}')
print(f'Total number of transactions: {sales_transactions_total:,}')

Timeframes

In [ ]:
list_of_timeframes = []

# Find the time difference
for location_id, df in tqdm(sales_and_menu_data.items()):

    # Find the time difference
    timedelta = df.index[-1] - df.index[0]

    # Convert to days, then to years
    years = timedelta.days / 365.25

    # Append
    list_of_timeframes.append(years)

# Turn into an np array for finding median, mean, and std
timeframes = np.array(list_of_timeframes)

# Display
print("Median: {:.2f} year range".format(np.median(timeframes)))
print("Mean: {:.2f} year range".format(np.mean(timeframes)))
print("SD: {:.2f} years".format(np.std(timeframes)))
print("Restaurants with less than a 2 year range: {}".format((timeframes < 2).sum()))

### Promotional Items

In [ ]:
promo_match_list = []

for loc_id, df in sales_and_menu_data.items():

    # Looking for promo
    promo_item = before_after_details.loc[loc_id, 'first_plant_based_mention']
    cross_over_date = before_after_details.loc[loc_id, 'cross_over_date']
    promo_df = fdf(df).filter('item_name', promo_item)
    ever_found = not promo_df.empty


    # Actual first
    plant_based = fdf(df).filter('is_plant_based', 'Yes')
    first_plant_based = plant_based['item_name'].iloc[0]
    its_date = plant_based.index[0]

    # Do they match?
    is_first = promo_item.lower() == first_plant_based.lower()

    row = {'location_id': loc_id, 'promo_item': promo_item, 'cross_over_date': cross_over_date, 'ever_found': ever_found, 'is_first': is_first, 'first_plant_based': first_plant_based, 'its_date': its_date}

    promo_match_list.append(row)
    
promo_match = pd.DataFrame(promo_match_list)

View

In [ ]:
promo_match

### Sales Visuals

In [ ]:
sales_and_menu_data['L69HYJ4Y3TR91'].columns

In [ ]:
sales_and_menu_data['L69HYJ4Y3TR91'].head(50)

In [ ]:
# items_tagged[items_tagged['item_name'].str.contains('[\[\]\(\)\{\}\\/\.\?\$\^\*\+]')]
items_tagged[items_tagged['item_name'].str.contains('[\[\]\{\}\\/\.\?\^\*\+]')]

In [ ]:
sales_and_menu_data['L69HYJ4Y3TR91']['item_modifications'].value_counts(dropna=False).index.tolist()

In [ ]:
sales_and_menu_data['L69HYJ4Y3TR91'].shape[0], sales_and_menu_data['L69HYJ4Y3TR91'].query('item_modifications != item_modifications')

In [ ]:
num_plots = 4#len(sales_and_menu_data)
cols = 4 
rows = num_plots

# Create a figure with multiple subplots
fig, axs = plt.subplots(rows, cols, figsize=(15, 5 * rows))
axs = axs.flatten()  # Flatten the array for easy indexing

# How many dishes to include
top_dishes_number = 20

# How many characters to include of each dish
num_characters = 22

# Lower the length of menu item names for the visual
items_tagged_copy = (items_tagged
        .assign(item_name=items_tagged.item_name.str.slice(0,num_characters).str.strip('.'))
        .drop_duplicates(subset=['location_id','item_name']))

for i, (location_id, data) in tqdm(enumerate(list(sales_and_menu_data.items())[0:4])):

    # Filter to the relevant restaurant
    items_tagged_res = items_tagged_copy.query('location_id == @location_id')

    # Just for the visual
    df = data.assign(item_name = data.item_name.str.slice(0,num_characters))
    
    # Colors for visual
    colors_df = pd.DataFrame(["#ff6961", "#aec6cf", "#77dd77"], columns=['colors'], index=['No','Unsure','Yes'])
    colors_df = pd.merge(items_tagged_res[['item_name', 'is_plant_based']], colors_df,
                  left_on='is_plant_based', right_index=True,
                  how='left')[['item_name','colors']]
    colors_df.set_index('item_name', drop=True, inplace=True)

    # Plant based
    plant_df = df[df['is_plant_based'] == 'Yes']

    # Get the top n items sorted in descending order
    top_items_by_times_ordered = df['item_name'].value_counts().nlargest(top_dishes_number).sort_values() # Times ordered
    top_items_by_quantity_ordered = df.groupby('item_name')['item_quantity'].sum().nlargest(top_dishes_number).sort_values() # Quantity ordered
    top_plant_based_items_by_times_ordered = plant_df['item_name'].value_counts().nlargest(top_dishes_number).sort_values() # Plant based, times ordered
    top_plant_based_items_by_quantity_ordered = plant_df.groupby('item_name')['item_quantity'].sum().nlargest(top_dishes_number).sort_values() # Plant based, quantity ordered

    dfs = [top_items_by_times_ordered, top_items_by_quantity_ordered, top_plant_based_items_by_times_ordered, top_plant_based_items_by_times_ordered]
    titles = ['Total by Times Ordered', 'Total by Quantity Ordered', 'Plant-Based by Times Ordered', 'Plant-Based by Quantity Ordered']
    xlabs = ['Times Ordered','Quantity Ordered','Times Ordered','Quantity Ordered']
    colors = ['blue','cyan', 'green', 'lime'] # currently not used

    to_visualize = zip(dfs, titles, xlabs, colors)

    for j, tup in enumerate(to_visualize):

        # Unpack
        df, title, xlab, color = tup

        # Overwrite to get custom colors within each graph
        colors = pd.merge(df, colors_df, left_index=True, right_index=True, how='left')['colors']

        colors = colors.fillna("#708086")

        # print(pd.concat([colors,df]))

        # Plot in the appropriate subplot as a horizontal bar chart
        axs[4*i + j].barh(df.index, df, color=colors)
        axs[4*i + j].set_title(f'{location_id}\n{title}')
        
        # Set y-axis label and make y-labels smaller
        axs[4*i + j].tick_params(axis='y', labelsize=10)
        axs[4*i + j].set_xlabel(xlab)

# Add a global title at the top of the figure
fig.suptitle('Distribution of Plant-Based and Total Item Orders in Each Restaurant', fontsize=16)

# Adjust layout and save the figure
plt.tight_layout()
plt.subplots_adjust(top=0.9)
# plt.subplots_adjust(top=0.97)  # Adjust the top margin to make room for the global title
plt.savefig('visuals/Restaurant Both Sales Distributions.png', bbox_inches='tight')